In [1]:
import numpy as np
import pandas as pd
import os
from PIL import Image
from tqdm import tqdm
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
import timm
from multiprocessing import Pool, cpu_count
import cv2
import warnings
import shutil
import random




/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

In [2]:
def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


seed_everything(42)

try:
    shutil.rmtree("/kaggle/working/test", ignore_errors=True)
    shutil.rmtree("/kaggle/working/train", ignore_errors=True)
except Exception:
    pass




In [3]:
def load_data(data_dir):
    test_csv = os.path.join(data_dir, "test.csv")
    test = pd.read_csv(test_csv)
    test_dir = os.path.join(data_dir, "test_images/")
    test["file_path"] = test["id_code"].map(
        lambda x: os.path.join(test_dir, f"{x}.png")
    )
    test["file_name"] = test["id_code"] + ".png"
    return test




In [4]:
data_dir = "/kaggle/input/aptos2019-blindness-detection/"
test_df = load_data(data_dir)




In [5]:
def crop_img(img, percentage):
    img_arr = np.array(img.convert("RGB"))
    img_gray = cv2.cvtColor(img_arr, cv2.COLOR_RGB2GRAY)

    nz = img_gray[img_gray != 0]
    if nz.size == 0:
        return Image.fromarray(img_arr)

    thr = 0.1 * float(np.mean(nz))
    threshold = img_gray > thr

    row_sums = np.sum(threshold, axis=1)
    col_sums = np.sum(threshold, axis=0)

    rows = np.where(row_sums > img_arr.shape[1] * percentage)[0]
    cols = np.where(col_sums > img_arr.shape[0] * percentage)[0]

    if rows.size == 0 or cols.size == 0:
        return Image.fromarray(img_arr)

    min_row, min_col = int(np.min(rows)), int(np.min(cols))
    max_row, max_col = int(np.max(rows)), int(np.max(cols))

    crop_arr = img_arr[min_row : max_row + 1, min_col : max_col + 1]
    return Image.fromarray(crop_arr)




In [6]:
def resize_maintain_aspect(img, desired_size):
    old_width, old_height = img.size
    if old_width == 0 or old_height == 0:
        img = img.convert("RGB")
        old_width, old_height = img.size

    aspect_ratio = old_width / old_height

    if aspect_ratio > 1:
        new_width = desired_size
        new_height = max(1, int(desired_size / aspect_ratio))
    else:
        new_height = desired_size
        new_width = max(1, int(desired_size * aspect_ratio))

    resized_img = img.convert("RGB").resize(
        (new_width, new_height), resample=Image.Resampling.LANCZOS
    )

    padded_image = Image.new("RGB", (desired_size, desired_size))
    x_offset = (desired_size - new_width) // 2
    y_offset = (desired_size - new_height) // 2
    padded_image.paste(resized_img, (x_offset, y_offset))

    return padded_image




In [7]:
def save_single(args):
    image_path, output_path_folder, percentage, output_size = args
    try:
        image = Image.open(image_path)
        croped_img = crop_img(image, percentage)
        image_resized = resize_maintain_aspect(croped_img, desired_size=output_size[0])

        output_image_path = os.path.basename(image_path)
        output_file_path = os.path.join(output_path_folder, output_image_path)
        image_resized.save(output_file_path)
    except Exception as e:
        return f"{image_path} -> {repr(e)}"
    return None




In [8]:
def fast_image_resize(df, output_path_folder, percentage, output_size=None):
    """Uses multiprocessing to make it fast"""
    if not output_size:
        warnings.warn("Need to specify output_size! For example: output_size=(100,100)")
        return

    if not os.path.exists(output_path_folder):
        os.makedirs(output_path_folder, exist_ok=True)

    jobs = []
    for df_item in range(len(df)):
        image_path = df.file_path.iloc[df_item]
        jobs.append((image_path, output_path_folder, percentage, output_size))

    processes = min(cpu_count(), 8)
    with Pool(processes=processes) as p:
        results = list(tqdm(p.imap_unordered(save_single, jobs), total=len(jobs)))
    errs = [r for r in results if r is not None]
    if len(errs) > 0:
        print(f"Resize warnings: {len(errs)} images failed. Example:\n{errs[0]}")




In [9]:
percentage = 0.01
resized_test_dir = "/kaggle/working/test/images_resized"
fast_image_resize(test_df, resized_test_dir, percentage, output_size=(224, 224))



100%|██████████| 367/367 [00:17<00:00, 21.39it/s]


In [10]:
missing = []
for fp in test_df["file_name"].tolist()[:10]:
    if not os.path.exists(os.path.join(resized_test_dir, fp)):
        missing.append(fp)
if missing:
    print("Warning: some resized images missing, e.g.:", missing[:3])




In [11]:
class BlindnessDataset(Dataset):
    def __init__(
        self, csv_file, root_dir, transform=None, test=False, fallback_root_dir=None
    ):
        self.annotations = pd.read_csv(csv_file)
        self.root_dir = root_dir
        self.fallback_root_dir = fallback_root_dir
        self.transform = transform
        self.test = test

    def __len__(self):
        return len(self.annotations)

    def __getitem__(self, idx):
        base = self.annotations.iloc[idx, 0] + ".png"
        img_name = os.path.join(self.root_dir, base)

        # Change (stability -> valid submission): if resized image missing, fall back to original.
        if (not os.path.exists(img_name)) and (self.fallback_root_dir is not None):
            fb = os.path.join(self.fallback_root_dir, base)
            if os.path.exists(fb):
                img_name = fb

        image = Image.open(img_name).convert("RGB")

        if self.transform:
            image = self.transform(image)

        if self.test:
            return image
        else:
            label = int(self.annotations.iloc[idx, 1])
            return image, label




In [12]:
transform = transforms.Compose(
    [
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ]
)



In [13]:
test_csv_file = "/kaggle/input/aptos2019-blindness-detection/test.csv"
test_root_dir = resized_test_dir
fallback_test_root_dir = "/kaggle/input/aptos2019-blindness-detection/test_images"

test_dataset = BlindnessDataset(
    test_csv_file,
    test_root_dir,
    transform=transform,
    test=True,
    fallback_root_dir=fallback_test_root_dir,
)
test_loader = DataLoader(
    test_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=2,
    pin_memory=torch.cuda.is_available(),
)



In [14]:
model_paths = {
    "inception_v4": "/kaggle/input/aptos_ensamble-models/pytorch/ensamble_v1/3/inception_v4.pth",
}

model_names = {
    "resnet18": "resnet18",
    "efficientnet_b5": "efficientnet_b5",
    "inception_resnet_v2": "inception_resnet_v2",
    "inception_v4": "inception_v4",
    "seresnext50_32x4d": "seresnext50_32x4d",
    "seresnext101_32x4d": "seresnext101_32x4d",
}



In [15]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
models_list = []
effective_model_keys = []

for model_key, path in model_paths.items():
    model_name = model_names[model_key]

    use_timm_pretrained = True
    if os.path.exists(path):
        use_timm_pretrained = False

    model = timm.create_model(model_name, pretrained=use_timm_pretrained, num_classes=5)

    if os.path.exists(path):
        state = torch.load(path, map_location="cpu")
        model.load_state_dict(state)
        print(f"Loaded weights for {model_key} from {path}")
    else:
        print(
            f"Warning: weights not found for {model_key} at {path}. Using timm pretrained weights."
        )

    model.to(device)
    model.eval()
    models_list.append(model)
    effective_model_keys.append(model_key)



model.safetensors:   0%|          | 0.00/171M [00:00<?, ?B/s]

In [16]:
validation_scores = {
    "inception_v4": 0.902,
}



In [17]:
total_score = sum(validation_scores.get(k, 1.0) for k in effective_model_keys)
weights = {k: validation_scores.get(k, 1.0) / total_score for k in effective_model_keys}



In [18]:
train_csv_file = "/kaggle/input/aptos2019-blindness-detection/train.csv"
train_dir = "/kaggle/input/aptos2019-blindness-detection/train_images"

train_df_full = pd.read_csv(train_csv_file)
rng = np.random.RandomState(42)
perm = rng.permutation(len(train_df_full))
val_size = max(1, int(0.1 * len(train_df_full)))
val_idx = perm[:val_size]
tr_idx = perm[val_size:]

train_df_split = train_df_full.iloc[tr_idx].reset_index(drop=True)
val_df_split = train_df_full.iloc[val_idx].reset_index(drop=True)

train_split_csv = "/kaggle/working/train_split.csv"
val_split_csv = "/kaggle/working/val_split.csv"
train_df_split.to_csv(train_split_csv, index=False)
val_df_split.to_csv(val_split_csv, index=False)

train_dataset = BlindnessDataset(
    train_split_csv,
    train_dir,
    transform=transform,
    test=False,
    fallback_root_dir=None,
)
val_dataset = BlindnessDataset(
    val_split_csv,
    train_dir,
    transform=transform,
    test=False,
    fallback_root_dir=None,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True,
    num_workers=2,
    pin_memory=torch.cuda.is_available(),
)
val_loader = DataLoader(
    val_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=2,
    pin_memory=torch.cuda.is_available(),
)



In [19]:
counts = (
    train_df_split["diagnosis"]
    .value_counts()
    .reindex([0, 1, 2, 3, 4], fill_value=0)
    .values.astype(np.float32)
)
counts = np.maximum(counts, 1.0)
inv = 1.0 / counts
ce_weights = inv / inv.sum() * 5.0  # normalized to keep loss scale similar
ce_weights_t = torch.tensor(ce_weights, dtype=torch.float32, device=device)
print("CE class weights:", ce_weights)




CE class weights: [0.21772279 1.018831   0.39487606 1.9743806  1.3941897 ]


In [20]:
def quadratic_weighted_kappa(y_true, y_pred, n_classes=5):
    y_true = np.asarray(y_true, dtype=int)
    y_pred = np.asarray(y_pred, dtype=int)
    assert y_true.shape == y_pred.shape
    O = np.zeros((n_classes, n_classes), dtype=np.float64)
    for a, b in zip(y_true, y_pred):
        if 0 <= a < n_classes and 0 <= b < n_classes:
            O[a, b] += 1.0
    act_hist = np.bincount(y_true, minlength=n_classes).astype(np.float64)
    pred_hist = np.bincount(y_pred, minlength=n_classes).astype(np.float64)
    E = np.outer(act_hist, pred_hist)
    if E.sum() > 0:
        E = E / E.sum() * O.sum()
    W = np.zeros((n_classes, n_classes), dtype=np.float64)
    for i in range(n_classes):
        for j in range(n_classes):
            W[i, j] = ((i - j) ** 2) / ((n_classes - 1) ** 2)
    num = (W * O).sum()
    den = (W * E).sum()
    return 1.0 - num / den if den > 0 else 0.0


def apply_thresholds(x, thresholds):
    t0, t1, t2, t3 = thresholds
    return np.where(
        x < t0, 0, np.where(x < t1, 1, np.where(x < t2, 2, np.where(x < t3, 3, 4)))
    ).astype(int)


def fit_thresholds(y_true, x_cont, n_classes=5, n_iter=3):
    y_true = np.asarray(y_true, dtype=int)
    x_cont = np.asarray(x_cont, dtype=np.float64)

    qs = np.quantile(x_cont, [0.2, 0.4, 0.6, 0.8])
    th = qs.astype(np.float64)

    best = quadratic_weighted_kappa(
        y_true, apply_thresholds(x_cont, th), n_classes=n_classes
    )

    xmin, xmax = float(np.min(x_cont)), float(np.max(x_cont))
    span = max(1e-6, xmax - xmin)
    _ = span  # keep variable to preserve semantics (no-op)

    for _ in range(n_iter):
        for k in range(4):
            lo = xmin if k == 0 else th[k - 1] + 1e-3
            hi = xmax if k == 3 else th[k + 1] - 1e-3
            if not (lo < hi):
                continue

            grid = np.linspace(lo, hi, 25)
            local_best = best
            local_th = th[k]
            for g in grid:
                cand = th.copy()
                cand[k] = g
                score = quadratic_weighted_kappa(
                    y_true, apply_thresholds(x_cont, cand), n_classes=n_classes
                )
                if score > local_best:
                    local_best = score
                    local_th = g
            th[k] = local_th
            best = local_best

    th = np.maximum.accumulate(th)
    th[1] = max(th[1], th[0] + 1e-3)
    th[2] = max(th[2], th[1] + 1e-3)
    th[3] = max(th[3], th[2] + 1e-3)
    return th, best




In [21]:
def get_head_params(m: nn.Module):
    params = []
    head = None

    if hasattr(m, "get_classifier"):
        try:
            head = m.get_classifier()
        except Exception:
            head = None

    if isinstance(head, nn.Module):
        params += list(head.parameters())
    else:
        for attr in ["classifier", "fc", "head"]:
            if hasattr(m, attr):
                hm = getattr(m, attr)
                if isinstance(hm, nn.Module):
                    params += list(hm.parameters())
                    break

    for attr in ["norm", "bn", "head_norm"]:
        if hasattr(m, attr):
            nm = getattr(m, attr)
            if isinstance(nm, nn.Module):
                params += list(nm.parameters())

    uniq = []
    seen = set()
    for p in params:
        if id(p) not in seen:
            uniq.append(p)
            seen.add(id(p))
    return uniq


# Change (score-up toward target): ensure the finetune actually updates the classifier head
# and that this same updated model is used later for thresholds + test inference (same semantics).
if len(models_list) > 0:
    model = models_list[0]
    model.train()

    for p in model.parameters():
        p.requires_grad = False

    head_params = get_head_params(model)
    for p in head_params:
        p.requires_grad = True

    if len(head_params) == 0:
        warnings.warn(
            "No classifier/head parameters found to fine-tune; finetuning will be skipped."
        )
    else:
        criterion = nn.CrossEntropyLoss(weight=ce_weights_t)

        optimizer = torch.optim.AdamW(
            filter(lambda p: p.requires_grad, model.parameters()),
            lr=3e-4,
            weight_decay=1e-4,
        )

        epochs = 1
        for epoch in range(epochs):
            running_loss = 0.0
            correct = 0
            total = 0
            pbar = tqdm(
                train_loader,
                total=len(train_loader),
                desc=f"finetune epoch {epoch+1}/{epochs}",
            )
            for images, labels in pbar:
                images = images.to(device, non_blocking=True)
                labels = labels.to(device, non_blocking=True)

                optimizer.zero_grad(set_to_none=True)
                logits = model(images)
                loss = criterion(logits, labels)
                loss.backward()
                optimizer.step()

                running_loss += float(loss.item()) * images.size(0)
                preds = torch.argmax(logits, dim=1)
                correct += int((preds == labels).sum().item())
                total += int(labels.numel())
                pbar.set_postfix(
                    loss=running_loss / max(1, total), acc=correct / max(1, total)
                )

        model.eval()
        models_list[0] = model



finetune epoch 1/1: 100%|██████████| 371/371 [01:58<00:00,  3.12it/s, acc=0.599, loss=1.38]


In [22]:
thresholds = np.array([0.5, 1.5, 2.5, 3.5], dtype=np.float64)

if len(models_list) > 0:
    val_probs = []
    val_true_list = []
    with torch.no_grad():
        for images, labels in tqdm(
            val_loader, total=len(val_loader), desc="val inference"
        ):
            images = images.to(device, non_blocking=True)
            out = nn.functional.softmax(models_list[0](images), dim=1)
            val_probs.append(out.detach().cpu().numpy())
            val_true_list.append(labels.detach().cpu().numpy())

    val_probs = np.concatenate(val_probs, axis=0)  # (N, 5)
    val_true = np.concatenate(val_true_list, axis=0).astype(int)

    val_cont = (val_probs * np.arange(5, dtype=np.float32)[None, :]).sum(axis=1)
    val_pred_argmax = np.argmax(val_probs, axis=1).astype(int)
    print("Val QWK (argmax):", quadratic_weighted_kappa(val_true, val_pred_argmax))

    thresholds, best_qwk = fit_thresholds(val_true, val_cont, n_classes=5, n_iter=3)
    print("Fitted thresholds:", thresholds)
    print("Val QWK (thresholded expected value):", best_qwk)
else:
    print("Warning: no models available; using default thresholds:", thresholds)



val inference: 100%|██████████| 21/21 [00:13<00:00,  1.58it/s]

Val QWK (argmax): 0.6757789653321369
Fitted thresholds: [1.2166507  1.80257282 2.0444231  2.87278907]
Val QWK (thresholded expected value): 0.8155663794972685


In [23]:
all_outputs = []

if len(models_list) == 0:
    raise RuntimeError("No models were initialized; cannot run test inference.")

with torch.no_grad():
    for images in tqdm(test_loader, total=len(test_loader), desc="test inference"):
        images = images.to(device, non_blocking=True)

        outputs = []
        for model_key, model in zip(effective_model_keys, models_list):
            out = nn.functional.softmax(model(images), dim=1)
            outputs.append(weights[model_key] * out.unsqueeze(0))

        outputs = torch.cat(outputs, dim=0)
        weighted_outputs = torch.sum(outputs, dim=0)
        all_outputs.append(weighted_outputs.detach().cpu().numpy())

all_outputs = np.concatenate(all_outputs, axis=0)

test_cont = (all_outputs * np.arange(5, dtype=np.float32)[None, :]).sum(axis=1)
test_cont = np.clip(test_cont, 0.0, 4.0)

final_predictions = apply_thresholds(test_cont, thresholds)



test inference: 100%|██████████| 23/23 [00:00<00:00, 24.16it/s]


In [24]:
test_ids = pd.read_csv(test_csv_file)["id_code"].values
assert len(test_ids) == len(final_predictions), (len(test_ids), len(final_predictions))

submission_df = pd.DataFrame({"id_code": test_ids, "diagnosis": final_predictions})

submission_path = "/kaggle/working/submission.csv"
submission_df.to_csv(submission_path, index=False)
print("Wrote:", submission_path)
print(submission_df.head())

Wrote: /kaggle/working/submission.csv
        id_code  diagnosis
0  b460ca9fa26f          0
1  6cee2e148520          0
2  ca6842bfcbc9          2
3  6cbc3dad809c          3
4  a9bc2f892cb3          0
